# Stable Diffusion WebUI Forge Neo + Cloudflare Tunnel

這份 notebook 會安裝 `Haoming02/sd-webui-forge-classic` 的 `neo` 分支。程式和 Python 環境放在 Colab 暫存空間，模型、設定、擴充套件與輸出則保存在 Google Drive。

> 執行前請在 **執行階段 → 變更執行階段類型** 選擇 GPU。第一次啟動會下載 Python 3.13、PyTorch 和相依套件，需要一些時間。

In [ ]:
#@title 1. 儲存空間設定
from pathlib import Path
import json
import os
import re

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
SHARED_DRIVE = "" #@param {type:"string"}
DRIVE_FOLDER = "sd/forge-neo" #@param {type:"string"}
USE_LEGACY_A1111_MODELS = True #@param {type:"boolean"}
LEGACY_A1111_FOLDER = "sd/stable-diffusion-webui" #@param {type:"string"}
AUTO_DETECT_COMFYUI = True #@param {type:"boolean"}
COMFYUI_PATHS = "" #@param {type:"string"}
# 多個 ComfyUI 根目錄或 models 目錄可用逗號、分號或換行分隔。

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/gdrive')
    drive_root = Path('/content/gdrive/Shareddrives') / SHARED_DRIVE if SHARED_DRIVE.strip() else Path('/content/gdrive/MyDrive')
    if SHARED_DRIVE.strip() and not drive_root.exists():
        raise FileNotFoundError(f'找不到共享雲端硬碟：{drive_root}')
    DATA_DIR = drive_root / DRIVE_FOLDER.strip('/')
    LEGACY_A1111_HOME = drive_root / LEGACY_A1111_FOLDER.strip('/')
else:
    DATA_DIR = Path('/content/forge-neo-data')
    LEGACY_A1111_HOME = Path('/content/nonexistent-a1111-home')

for relative in ('models/Stable-diffusion', 'models/Lora', 'models/VAE', 'models/ControlNet', 'models/text_encoder', 'extensions', 'output'):
    (DATA_DIR / relative).mkdir(parents=True, exist_ok=True)

print(f'Forge Neo 資料目錄：{DATA_DIR}')
if USE_LEGACY_A1111_MODELS and LEGACY_A1111_HOME.exists():
    print(f'也會讀取舊 A1111 模型：{LEGACY_A1111_HOME}')
elif USE_LEGACY_A1111_MODELS:
    print(f'找不到舊 A1111 目錄，將略過：{LEGACY_A1111_HOME}')

COMFY_REFERENCE_PATH = Path('/content/forge-neo-comfy-paths.yaml')

def build_comfy_reference_config():
    candidates = []
    if AUTO_DETECT_COMFYUI:
        candidates.extend([Path('/content/ComfyUI'), Path('/content/comfyui')])
        if USE_GOOGLE_DRIVE:
            candidates.extend([drive_root / 'ComfyUI', drive_root / 'comfyui', drive_root / 'sd/ComfyUI'])
    candidates.extend(Path(value.strip()) for value in re.split(r'[,;\n]+', COMFYUI_PATHS) if value.strip())

    model_roots = []
    seen = set()
    for candidate in candidates:
        candidate = Path(os.path.expandvars(str(candidate))).expanduser()
        model_root = candidate / 'models' if (candidate / 'models').is_dir() else candidate
        if not model_root.is_dir():
            if str(candidate) in [value.strip() for value in re.split(r'[,;\n]+', COMFYUI_PATHS) if value.strip()]:
                print(f'找不到 ComfyUI 路徑，將略過：{candidate}')
            continue
        key = str(model_root.resolve())
        if key not in seen:
            seen.add(key)
            model_roots.append(model_root)

    if not model_roots:
        if COMFY_REFERENCE_PATH.exists():
            COMFY_REFERENCE_PATH.unlink()
        print('沒有偵測到 ComfyUI 模型目錄。')
        return None

    config = {}
    for index, model_root in enumerate(model_roots, start=1):
        config[f'comfy_{index}'] = {
            'base_path': str(model_root),
            'checkpoints': 'checkpoints',
            'diffusion_models': 'diffusion_models',
            'unet': 'unet',
            'clip': 'clip',
            'text_encoders': 'text_encoders',
            'loras': 'loras',
            'vae': 'vae',
            'controlnet': 'controlnet',
        }
    COMFY_REFERENCE_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Forge Neo 將讀取下列 ComfyUI 模型目錄：')
    for model_root in model_roots:
        print(f'  - {model_root}')
    return COMFY_REFERENCE_PATH

COMFY_REFERENCE_FILE = build_comfy_reference_config()

In [ ]:
#@title 2. 安裝或更新 Forge Neo
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Haoming02/sd-webui-forge-classic.git'
REPO_BRANCH = 'neo'
REPO_DIR = Path('/content/sd-webui-forge-neo')
VENV_DIR = REPO_DIR / 'venv'
VENV_PYTHON = VENV_DIR / 'bin/python'

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('找不到 NVIDIA GPU。請先把 Colab 執行階段切換為 GPU。')
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} 已存在但不是 Git repository，請重新啟動執行階段。')
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if not VENV_PYTHON.exists():
    subprocess.run(['uv', 'venv', str(VENV_DIR), '--python', '3.13', '--seed'], check=True)

version = subprocess.check_output([str(VENV_PYTHON), '--version'], text=True).strip()
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Forge Neo {commit} 已準備完成（{version}）')

## 3. 模型下載（選用）

如果模型已經在 Google Drive，可以略過。`MODEL_TYPE` 會決定儲存的子目錄；私人 Hugging Face 或 Civitai 下載可填 Token。

In [ ]:
#@title 下載一個模型
MODEL_URL = "" #@param {type:"string"}
MODEL_FILENAME = "" #@param {type:"string"}
MODEL_TYPE = "Checkpoint" #@param ["Checkpoint", "LoRA", "VAE", "ControlNet", "Text Encoder"]
HF_TOKEN = "" #@param {type:"string"}
CIVITAI_TOKEN = "" #@param {type:"string"}

if MODEL_URL.strip():
    import requests
    from urllib.parse import unquote, urlparse

    model_folders = {
        'Checkpoint': 'Stable-diffusion',
        'LoRA': 'Lora',
        'VAE': 'VAE',
        'ControlNet': 'ControlNet',
        'Text Encoder': 'text_encoder',
    }
    target_dir = DATA_DIR / 'models' / model_folders[MODEL_TYPE]
    parsed = urlparse(MODEL_URL.strip())
    filename = MODEL_FILENAME.strip() or Path(unquote(parsed.path)).name
    if not filename:
        raise ValueError('無法從網址判斷檔名，請填寫 MODEL_FILENAME。')
    target = target_dir / filename

    if target.exists():
        print(f'檔案已存在：{target}')
    elif parsed.netloc.lower() == 'drive.google.com':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        import gdown
        if not gdown.download(MODEL_URL.strip(), str(target), quiet=False, fuzzy=True):
            raise RuntimeError('Google Drive 下載失敗。')
    else:
        headers = {'User-Agent': 'Forge-Neo-Colab'}
        if 'huggingface.co' in parsed.netloc and HF_TOKEN.strip():
            headers['Authorization'] = f'Bearer {HF_TOKEN.strip()}'
        if 'civitai.com' in parsed.netloc and CIVITAI_TOKEN.strip():
            headers['Authorization'] = f'Bearer {CIVITAI_TOKEN.strip()}'
        partial = target.with_name(target.name + '.part')
        with requests.get(MODEL_URL.strip(), headers=headers, stream=True, timeout=60) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            downloaded = 0
            with partial.open('wb') as output:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    output.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        print(f'\r下載中：{downloaded / total:.1%}', end='')
        partial.replace(target)
        print()
    print(f'模型位置：{target}')
else:
    print('MODEL_URL 留空，略過下載。')

## 4. 第三方擴充套件（選用）

Forge Neo 已內建 ControlNet。Notebook 預設安裝 Civitai Helper RED UPDATE；其他擴充套件請只加入已確認支援 Forge Neo 的 repository，每行一個 Git URL。

In [ ]:
#@title 安裝或更新擴充套件
INSTALL_CIVITAI_HELPER = True #@param {type:"boolean"}
EXTENSION_URLS = "" #@param {type:"string"}

import re
from urllib.parse import urlparse

CIVITAI_HELPER_URL = 'https://github.com/Replactionap/Stable-Diffusion-Webui-Civitai-Helper-RED-UPDATE.git'
extensions_dir = DATA_DIR / 'extensions'
extension_urls = [line.strip() for line in EXTENSION_URLS.splitlines() if line.strip()]
if INSTALL_CIVITAI_HELPER:
    extension_urls.insert(0, CIVITAI_HELPER_URL)

for extension_url in dict.fromkeys(extension_urls):
    repo_name = Path(urlparse(extension_url).path).name.removesuffix('.git')
    if not repo_name or not re.fullmatch(r'[A-Za-z0-9._-]+', repo_name):
        raise ValueError(f'無法判斷安全的擴充套件目錄名稱：{extension_url}')
    destination = extensions_dir / repo_name
    if (destination / '.git').exists():
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)
    elif destination.exists():
        print(f'略過已存在但不是 Git repository 的目錄：{destination}')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', extension_url, str(destination)], check=True)

print('擴充套件處理完成。')

In [ ]:
#@title 5. 啟動 Forge Neo 與 Cloudflare Tunnel
WEBUI_USER = "" #@param {type:"string"}
WEBUI_PASSWORD = "" #@param {type:"string"}
CF_TUNNEL_TOKEN = "" #@param {type:"string"}
ENABLE_API = False #@param {type:"boolean"}
INSTALL_XFORMERS = False #@param {type:"boolean"}
EXTRA_ARGS = "" #@param {type:"string"}

import re
import shlex
import stat
import threading
import urllib.request

PORT = 7860
CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    print('下載 cloudflared...')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CLOUDFLARED)
    CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

previous_tunnel = globals().get('cloudflared_process')
if previous_tunnel is not None and previous_tunnel.poll() is None:
    previous_tunnel.terminate()
    previous_tunnel.wait(timeout=10)

if CF_TUNNEL_TOKEN.strip():
    tunnel_command = [str(CLOUDFLARED), 'tunnel', '--no-autoupdate', 'run', '--token', CF_TUNNEL_TOKEN.strip()]
    print('正在啟動具名 Cloudflare Tunnel；請使用你在 Cloudflare 設定的 Public Hostname。')
else:
    tunnel_command = [str(CLOUDFLARED), 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}']
    print('正在建立 Cloudflare Quick Tunnel...')

cloudflared_process = subprocess.Popen(
    tunnel_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def show_tunnel_output(process):
    announced = False
    for line in process.stdout:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match and not announced:
            announced = True
            print(f'\nCloudflare 網址：{match.group(0)}\n')
        elif ' ERR ' in line or ' error' in line.lower():
            print('[cloudflared]', line.rstrip())

threading.Thread(target=show_tunnel_output, args=(cloudflared_process,), daemon=True).start()

launch_args = [
    str(VENV_PYTHON), str(REPO_DIR / 'launch.py'),
    '--uv', '--listen', '--port', str(PORT),
    '--data-dir', str(DATA_DIR),
]
if ENABLE_API:
    launch_args.append('--api')
if INSTALL_XFORMERS:
    launch_args.append('--xformers')
if USE_LEGACY_A1111_MODELS and LEGACY_A1111_HOME.exists():
    launch_args.extend(['--forge-ref-a1111-home', str(LEGACY_A1111_HOME)])
COMFY_REFERENCE_FILE = build_comfy_reference_config()
if COMFY_REFERENCE_FILE is not None:
    launch_args.extend(['--forge-ref-comfy-yaml', str(COMFY_REFERENCE_FILE)])
if WEBUI_USER.strip() and WEBUI_PASSWORD:
    credentials = f'{WEBUI_USER.strip()}:{WEBUI_PASSWORD}'
    launch_args.extend(['--gradio-auth', credentials])
    if ENABLE_API:
        launch_args.extend(['--api-auth', credentials])
elif WEBUI_USER.strip() or WEBUI_PASSWORD:
    raise ValueError('WEBUI_USER 與 WEBUI_PASSWORD 必須一起設定。')
if ENABLE_API and not (WEBUI_USER.strip() and WEBUI_PASSWORD):
    print('警告：API 已啟用，但沒有設定 API 驗證。')
launch_args.extend(shlex.split(EXTRA_ARGS))

print(f'啟動 Forge Neo，資料目錄：{DATA_DIR}')
try:
    subprocess.run(launch_args, cwd=REPO_DIR, check=True)
finally:
    if cloudflared_process.poll() is None:
        cloudflared_process.terminate()

## 常見問題

- **啟動時顯示找不到 GPU**：確認 Colab 執行階段已選 GPU，然後從第一格重新執行。
- **擴充套件導致錯誤**：刪除 Google Drive 中 `forge-neo/extensions` 內對應的擴充套件目錄。
- **找不到舊模型**：確認 `LEGACY_A1111_FOLDER` 是相對於 MyDrive 或所選共享雲端硬碟的路徑。
- **找不到 ComfyUI 模型**：在 `COMFYUI_PATHS` 填入 ComfyUI 根目錄或其 `models` 目錄；多個路徑以逗號、分號或換行分隔。
- **Cloudflare 網址無法開啟**：等待 WebUI 顯示 `Running on local URL`；Quick Tunnel 網址會先於 WebUI 準備完成。
- **套件或環境損壞**：重新啟動 Colab 執行階段，再依序執行 notebook。